# DisasterM3 — U-Net Segmentation Fine-Tuning
## Member A — Task A3: Copy-Paste Augmentation (run against plain baseline, sampler OFF)

**Run order in this notebook:** Config → Load data → Dataset/DataLoader (plain `shuffle=True`) →
Loss → A3 offline instance extraction → A3 CopyPasteAugDataset wrapper → rebuild `train_loader`
with copy-paste dataset → reset run state → resume-check → train 10 epochs → final per-class
metrics → save results as `*_copypaste`.

This is a full **Run All** notebook — GPU (T4) must be attached (check Cell 2's printed
`Device:` line says `Tesla T4`, not `CPU`, before letting it run long).


In [2]:
# ── Cell 1: Environment Setup (Run-All safe) ──────────────────────────────
# Mirrors train_qwen_disasterm3.ipynb Cell 1: pinned versions, idempotent,
# halts on first install to force kernel restart.
import importlib.metadata as _md

PINS = {
    "torch": "2.4.1",
    "torchvision": "0.19.1",
    "torchaudio": "2.4.1",
    "segmentation-models-pytorch": "0.3.4",
    "albumentations": "1.4.21",
    "opencv-python-headless": "4.10.0.84",
}

_mismatched = []
for _pkg, _want in PINS.items():
    try:
        _have = _md.version(_pkg)
    except _md.PackageNotFoundError:
        _have = "not installed"
    if _have != _want:
        _mismatched.append(f"{_pkg}: {_have} → {_want}")

if _mismatched:
    print("⏳ Installing pinned versions:")
    for _m in _mismatched:
        print(f"   {_m}")
    !pip install -q \
        torch==2.4.1 \
        torchvision==0.19.1 \
        torchaudio==2.4.1 \
        segmentation-models-pytorch==0.3.4 \
        albumentations==1.4.21 \
        opencv-python-headless==4.10.0.84 \
        pillow \
        huggingface_hub[hf_xet]
    raise SystemExit(
        "✓ Dependencies installed. RESTART THE KERNEL NOW "
        "(Run → Restart & clear cell outputs), then click 'Run All' again — "
        "this cell will detect the correct versions and skip installation."
    )

print("✓ All pinned versions already installed — proceeding.")

✓ All pinned versions already installed — proceeding.


In [3]:
# ── Cell 2: Configuration ─────────────────────────────────────────────────
import os
import json
import torch
from pathlib import Path
from datetime import datetime

# ── Paths (same DisasterM3 mirror dataset mount as VLM notebook) ──
DATA_ROOT = Path("/kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct")
MANIFEST_PATH = DATA_ROOT / "train_release.json"

# ── Model ──
ENCODER_NAME = "resnet34"     # ImageNet-pretrained backbone
ENCODER_WEIGHTS = "imagenet"  # Transfer learning
NUM_CLASSES = 4               # 0=Background, 1=Intact, 2=Damaged, 3=Destroyed

# ── Training config ──
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 10                # A3 comparison run: 10 epochs, same as baseline/A2
BATCH_SIZE = 16
NUM_WORKERS = 2
IMAGE_SIZE = 512

# ── Checkpointing (12-hour Kaggle session cap) ──
CHECKPOINT_DIR = "/kaggle/working/unet_checkpoints"
SAVE_EVERY_N_EPOCHS = 2
TIME_LIMIT_HOURS = 11.4

# ── Cross-session resume ──
HF_CHECKPOINT_REPO = None
RESUME_EPOCH = 0

# ── Output ──
OUTPUT_DIR = "/kaggle/working/unet_disasterm3"
MODEL_NAME = f"disasterm3_unet_{ENCODER_NAME}_ep{NUM_EPOCHS}"

print(f"\u2713 Config loaded")
print(f"  Encoder: {ENCODER_NAME} (pretrained={ENCODER_WEIGHTS})")
print(f"  Classes: {NUM_CLASSES} (Background, Intact, Damaged, Destroyed)")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Image size: {IMAGE_SIZE}\u00d7{IMAGE_SIZE}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")


✓ Config loaded
  Encoder: resnet34 (pretrained=imagenet)
  Classes: 4 (Background, Intact, Damaged, Destroyed)
  Batch size: 16
  Image size: 512×512
  Epochs: 10
  Device: Tesla T4
  VRAM: 15.6 GB


## 2. Load & Prepare Segmentation Data
Builds the combined multi-class masks (0=Background, 1=Intact, 2=Damaged, 3=Destroyed)
from the three binary source mask folders.

In [4]:
# ── Cell 4: Build combined multi-class masks from 3 binary sources ──
import cv2
import numpy as np
import os
from collections import Counter, defaultdict
from pathlib import Path
import json

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)
print(f"Loaded {len(raw_data):,} entries from manifest")

# ── Filter: ONLY Building Damage Assessment segmentation entries ──
bda_entries = [
    e for e in raw_data
    if e.get("task") == "Referring Expression Segmentation"
    and e.get("cls_description") == "Building Damage Assessment"
]
print(f"Building Damage Assessment entries: {len(bda_entries):,}")

# ── Path resolution ──
def resolve_path(rel_path):
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    candidates = [
        DATA_ROOT / rel_path,
        DATA_ROOT / "train_images" / rel_path,
        DATA_ROOT / "train_images" / "train_images" / filename,
        DATA_ROOT / "masks" / rel_path,
        DATA_ROOT / "masks" / "masks" / filename,
    ]
    for c in candidates:
        if c.exists():
            return str(c)
    return None

# ── Group entries by underlying post-disaster image ──
by_image = defaultdict(dict)

for e in bda_entries:
    post_img = e.get("post_image_path", "")
    if e.get("image_type") != "Optical":
        continue
    mask_rel = e.get("ground_truth", "")
    folder = Path(mask_rel.replace("\\", "/")).parent.name
    resolved = resolve_path(mask_rel)
    if resolved:
        by_image[post_img][folder] = resolved
        by_image[post_img]["post_image_path"] = post_img

print(f"Unique images with at least one mask: {len(by_image):,}")

# ── Build combined (image, mask) pairs ──
FOLDER_TO_CLASS = {
    "train_building_intact_mask": 1,
    "train_building_damaged_mask": 2,
    "train_building_destroyed_mask": 3,
}

os.makedirs("/tmp/combined_masks", exist_ok=True)

pairs = []
skipped_img = 0

for post_img, mask_dict in by_image.items():
    img_path = resolve_path(post_img)
    if not img_path:
        skipped_img += 1
        continue

    combined_mask = None
    for folder, class_id in FOLDER_TO_CLASS.items():
        mask_path = mask_dict.get(folder)
        if mask_path is None:
            continue
        m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if m is None:
            continue
        if combined_mask is None:
            combined_mask = np.zeros_like(m, dtype=np.uint8)
        combined_mask[m > 0] = class_id

    if combined_mask is None:
        continue

    mask_filename = Path(img_path).name
    save_path = f"/tmp/combined_masks/{mask_filename}"
    cv2.imwrite(save_path, combined_mask)

    pairs.append({
        "image_path": img_path,
        "mask_path": save_path,
    })

print(f"\n\u2713 Built {len(pairs):,} valid (image, mask) pairs")
print(f"  Skipped (image not found): {skipped_img:,}")

if pairs:
    print(f"\n  Sample: {pairs[0]['image_path']}")
    print(f"  Mask saved at: {pairs[0]['mask_path']}")


Loaded 92,968 entries from manifest
Building Damage Assessment entries: 14,531
Unique images with at least one mask: 6,443

✓ Built 6,443 valid (image, mask) pairs
  Skipped (image not found): 0

  Sample: /kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct/train_images/train_images/bata_explosion_post_0.png
  Mask saved at: /tmp/combined_masks/bata_explosion_post_0.png


In [5]:
# ── Cell 5: PyTorch Dataset with augmentations ────────────────────────────
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class DisasterM3SegDataset(Dataset):
    """Dataset for DisasterM3 segmentation entries.

    Loads post-disaster images and their pre-merged combined damage masks.
      0 = Background
      1 = Intact (no damage)
      2 = Damaged
      3 = Destroyed
    """

    def __init__(self, pairs, transform=None, image_size=512):
        self.pairs = pairs
        self.transform = transform
        self.image_size = image_size

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pair = self.pairs[idx]

        image = cv2.imread(pair["image_path"])
        if image is None:
            return self.__getitem__((idx + 1) % len(self))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(pair["mask_path"], cv2.IMREAD_GRAYSCALE)
        if mask is None:
            return self.__getitem__((idx + 1) % len(self))

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        return image, mask.long()


# ── Augmentation pipelines ──
train_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

# ── Train/Val split (90/10) — SAME random_state as baseline/A2 for a fair comparison ──
from sklearn.model_selection import train_test_split

train_pairs, val_pairs = train_test_split(pairs, test_size=0.1, random_state=42)

train_dataset = DisasterM3SegDataset(train_pairs, transform=train_transform, image_size=IMAGE_SIZE)
val_dataset = DisasterM3SegDataset(val_pairs, transform=val_transform, image_size=IMAGE_SIZE)

# NOTE: plain shuffle=True here on purpose — A3 isolates copy-paste as the only variable,
# so the sampler from A2 is intentionally NOT used in this notebook.
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"\u2713 Datasets created")
print(f"  Train: {len(train_dataset):,} samples ({len(train_loader):,} batches)")
print(f"  Val:   {len(val_dataset):,} samples ({len(val_loader):,} batches)")


/usr/local/lib/python3.12/dist-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.21). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


✓ Datasets created
  Train: 5,798 samples (363 batches)
  Val:   645 samples (41 batches)


## 3. Task A3 — Copy-Paste Augmentation
**Step 1:** extract every Damaged/Destroyed building instance from the training masks (run once).
**Step 2:** wrap `train_dataset` in `CopyPasteAugDataset`, which pastes 1–3 random instances onto
each training image with 50% probability. **Step 3:** rebuild `train_loader` from the wrapped
dataset, keeping `shuffle=True` (no sampler) so copy-paste is the only variable vs. baseline.

In [6]:
# ── A3 Step 1: Extract minority-class building instances (run ONCE) ──
import cv2
import numpy as np
import os

INSTANCE_DIR = "/kaggle/working/minority_instances"
os.makedirs(f"{INSTANCE_DIR}/damaged", exist_ok=True)
os.makedirs(f"{INSTANCE_DIR}/destroyed", exist_ok=True)

instance_count = {2: 0, 3: 0}

for pair in train_pairs:  # FIX: train-only, prevents val leakage into copy-paste bank
    img = cv2.imread(pair["image_path"])
    mask = cv2.imread(pair["mask_path"], cv2.IMREAD_GRAYSCALE)
    if img is None or mask is None:
        continue

    for cls_id, cls_name in [(2, "damaged"), (3, "destroyed")]:
        binary = (mask == cls_id).astype(np.uint8)
        if binary.sum() == 0:
            continue

        num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary)
        for label_id in range(1, num_labels):
            area = stats[label_id, cv2.CC_STAT_AREA]
            if area < 100:  # skip tiny noise blobs
                continue

            x = stats[label_id, cv2.CC_STAT_LEFT]
            y = stats[label_id, cv2.CC_STAT_TOP]
            w = stats[label_id, cv2.CC_STAT_WIDTH]
            h = stats[label_id, cv2.CC_STAT_HEIGHT]

            inst_img = img[y:y+h, x:x+w].copy()
            inst_mask = (labels[y:y+h, x:x+w] == label_id).astype(np.uint8)

            save_id = instance_count[cls_id]
            cv2.imwrite(f"{INSTANCE_DIR}/{cls_name}/img_{save_id}.png", inst_img)
            cv2.imwrite(f"{INSTANCE_DIR}/{cls_name}/mask_{save_id}.png", inst_mask * 255)
            instance_count[cls_id] += 1

print(f"Extracted instances \u2014 Damaged: {instance_count[2]}, Destroyed: {instance_count[3]}")


Extracted instances — Damaged: 40704, Destroyed: 16682


In [7]:
# ── A3 Step 2: Copy-Paste Augmentation Dataset Wrapper ──
import random, glob, torch
from torch.utils.data import Dataset

class CopyPasteAugDataset(Dataset):
    """Wraps base dataset. Randomly pastes minority-class instances onto images."""

    def __init__(self, base_dataset, instance_dir, paste_prob=0.5, max_paste=3):
        self.base = base_dataset
        self.paste_prob = paste_prob
        self.max_paste = max_paste

        self.instances = []
        for cls_id, cls_name in [(2, "damaged"), (3, "destroyed")]:
            img_paths = sorted(glob.glob(f"{instance_dir}/{cls_name}/img_*.png"))
            for img_path in img_paths:
                mask_path = img_path.replace("img_", "mask_")
                inst_img = cv2.imread(img_path)
                inst_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                if inst_img is not None and inst_mask is not None:
                    inst_mask = (inst_mask > 127).astype(np.uint8)
                    self.instances.append((inst_img, inst_mask, cls_id))

        print(f"Loaded {len(self.instances)} minority instances for Copy-Paste")

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        image, mask = self.base[idx]

        if random.random() < self.paste_prob and self.instances:
            n = random.randint(1, self.max_paste)
            img_np = image.permute(1, 2, 0).numpy()  # [H, W, C]
            mask_np = mask.numpy()                    # [H, W]

            for _ in range(n):
                inst_img, inst_mask, cls_id = random.choice(self.instances)
                ih, iw = inst_mask.shape[:2]

                if ih >= img_np.shape[0] or iw >= img_np.shape[1]:
                    continue

                y = random.randint(0, img_np.shape[0] - ih)
                x = random.randint(0, img_np.shape[1] - iw)
                paste_mask = inst_mask > 0

                # inst_img is BGR from cv2.imread; base dataset images are RGB+normalized,
                # so convert BGR->RGB before normalizing to match.
                inst_normalized = inst_img[..., ::-1].astype(np.float32) / 255.0
                inst_normalized = (inst_normalized - np.array([0.485, 0.456, 0.406])) / np.array([0.229, 0.224, 0.225])

                img_np[y:y+ih, x:x+iw][paste_mask] = inst_normalized[paste_mask]
                mask_np[y:y+ih, x:x+iw][paste_mask] = cls_id

            image = torch.from_numpy(img_np).permute(2, 0, 1).float()
            mask = torch.from_numpy(mask_np).long()

        return image, mask


# ── A3 Step 3: wrap train_dataset and rebuild train_loader (still shuffle=True, no sampler) ──
train_dataset_cp = CopyPasteAugDataset(train_dataset, INSTANCE_DIR, paste_prob=0.5, max_paste=3)

train_loader = DataLoader(
    train_dataset_cp,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print(f"\u2713 train_loader rebuilt with CopyPasteAugDataset ({len(train_dataset_cp):,} samples)")


Loaded 57386 minority instances for Copy-Paste
✓ train_loader rebuilt with CopyPasteAugDataset (5,798 samples)


## 4. Loss Function (unchanged from baseline/A2 — same architecture/loss for a fair comparison)

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

NUM_CLASSES = 4  # 0=Background, 1=Intact, 2=Damaged, 3=Destroyed

# ── Class weights computed from your pixel counts (Cell 4 output) ──
class_pixel_counts = torch.tensor([6267276585, 395903867, 70675310, 22119406], dtype=torch.float)
total_pixels = class_pixel_counts.sum()
class_weights = total_pixels / (NUM_CLASSES * class_pixel_counts)
class_weights = class_weights / class_weights.sum() * NUM_CLASSES  # normalize so mean weight \u2248 1
print("Class weights:", class_weights.tolist())

class DiceLoss(nn.Module):
    """Multi-class Dice loss. Expects logits [B, C, H, W] and integer targets [B, H, W]."""
    def __init__(self, num_classes, smooth=1e-5):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=1)
        targets_onehot = F.one_hot(targets, self.num_classes).permute(0, 3, 1, 2).float()

        dims = (0, 2, 3)
        intersection = torch.sum(probs * targets_onehot, dims)
        cardinality = torch.sum(probs + targets_onehot, dims)
        dice_per_class = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)

        return 1.0 - dice_per_class.mean()

class CombinedLoss(nn.Module):
    """Weighted CrossEntropy + Dice, summed with configurable weighting."""
    def __init__(self, class_weights, num_classes, ce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=class_weights)
        self.dice = DiceLoss(num_classes)
        self.ce_weight = ce_weight
        self.dice_weight = dice_weight

    def forward(self, logits, targets):
        ce_loss = self.ce(logits, targets)
        dice_loss = self.dice(logits, targets)
        return self.ce_weight * ce_loss + self.dice_weight * dice_loss, ce_loss.item(), dice_loss.item()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = CombinedLoss(class_weights.to(DEVICE), NUM_CLASSES)
print(f"\u2713 Loss function ready on {DEVICE}")


Class weights: [0.010286855511367321, 0.16284401714801788, 0.912207841873169, 2.914661169052124]
✓ Loss function ready on cuda


In [9]:
# ── Cell 7: Resume from HF checkpoint (mirrors VLM notebook Cell 8.5) ────
# Cross-session resume: download the latest checkpoint from HF Hub.
start_epoch = 0

if HF_CHECKPOINT_REPO:
    from huggingface_hub import hf_hub_download
    from kaggle_secrets import UserSecretsClient

    hf_token = UserSecretsClient().get_secret("HF_TOKEN")

    try:
        ckpt_path = hf_hub_download(
            repo_id=HF_CHECKPOINT_REPO,
            filename=f"unet_epoch_{RESUME_EPOCH}.pth",
            token=hf_token,
        )
        checkpoint = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        start_epoch = checkpoint["epoch"] + 1
        print(f"\u2713 Resumed from HF checkpoint: epoch {checkpoint['epoch']}")
        print(f"  Starting at epoch {start_epoch}")
    except Exception as e:
        print(f"\u26a0 Could not load HF checkpoint: {e}")
        print("  Starting fresh.")
else:
    if os.path.exists(CHECKPOINT_DIR):
        ckpts = sorted(
            [f for f in os.listdir(CHECKPOINT_DIR) if f.endswith(".pth")],
            key=lambda x: int(x.split("_")[-1].split(".")[0])
        )
        if ckpts:
            latest = os.path.join(CHECKPOINT_DIR, ckpts[-1])
            checkpoint = torch.load(latest, map_location=DEVICE)
            model.load_state_dict(checkpoint["model_state_dict"])
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
            scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
            start_epoch = checkpoint["epoch"] + 1
            print(f"\u2713 Resumed from local checkpoint: {latest}")
            print(f"  Starting at epoch {start_epoch}")
        else:
            print("No checkpoints found \u2014 starting fresh.")
    else:
        print("No checkpoint directory \u2014 starting fresh.")


No checkpoint directory — starting fresh.


**Note:** Cell 7 references `model`/`optimizer`/`scheduler`, which are created in the
training cell below. On Kaggle's normal top-to-bottom "Run All", the training cell defines
`model`/`optimizer`/`scheduler` *before* the `if HF_CHECKPOINT_REPO:` branch would ever need
them — since `HF_CHECKPOINT_REPO = None` here, this cell just sets `start_epoch = 0` and exits
early without touching those variables, so the ordering is safe. If you ever set
`HF_CHECKPOINT_REPO`, move this cell to *after* the model/optimizer/scheduler are created.

In [10]:
import time
import os
import datetime
import json
import numpy as np
import segmentation_models_pytorch as smp
from tqdm import tqdm
from IPython.display import display, HTML
from torch.cuda.amp import GradScaler, autocast

training_start = time.time()
epoch_times = []

# ── Model ──
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=NUM_CLASSES,
).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
scaler = GradScaler()  # AMP Optimization

def compute_iou(pred, target, num_classes):
    ious = []
    for cls in range(num_classes):
        pred_cls = (pred == cls)
        target_cls = (target == cls)
        intersection = (pred_cls & target_cls).sum().item()
        union = (pred_cls | target_cls).sum().item()
        if union == 0:
            ious.append(float('nan'))
        else:
            ious.append(intersection / union)
    return ious

def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    total_loss = 0.0
    for images, masks in tqdm(loader, desc="Train", leave=False):
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()

        with autocast():
            logits = model(images)
            loss, ce_val, dice_val = criterion(logits, masks)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_ious = [[] for _ in range(NUM_CLASSES)]

    for images, masks in tqdm(loader, desc="Val", leave=False):
        images, masks = images.to(device), masks.to(device)

        with autocast():
            logits = model(images)
            loss, _, _ = criterion(logits, masks)

        total_loss += loss.item()
        preds = logits.argmax(dim=1)

        for pred, mask in zip(preds, masks):
            ious = compute_iou(pred.cpu(), mask.cpu(), NUM_CLASSES)
            for cls, iou in enumerate(ious):
                if not np.isnan(iou):
                    all_ious[cls].append(iou)

    class_ious = [np.mean(cls_ious) if cls_ious else 0.0 for cls_ious in all_ious]
    miou = np.mean([iou for iou in class_ious if iou > 0])
    return total_loss / len(loader), miou

# ── Kaggle Safety Wall & Checkpointing Setup ──
TIME_LIMIT_HOURS = 11.5
deadline = time.time() + TIME_LIMIT_HOURS * 3600
os.makedirs("/kaggle/working/checkpoints", exist_ok=True)

# ── Force-reset run state for THIS experiment (A3) — do not inherit best_miou from a prior run ──
best_miou = 0.0
history = {"train_loss": [], "val_loss": [], "miou": []}

# ── UI Setup ──
progress_html = display(HTML(f"<div><progress value='0' max='{NUM_EPOCHS}' style='width:300px; height:20px; vertical-align: middle;'></progress> [0/{NUM_EPOCHS}]</div>"), display_id=True)
table_html = display(HTML("<table border='1' class='dataframe'><thead><tr style='text-align: left;'><th>Epoch</th><th>Train Loss</th><th>Val Loss</th><th>mIoU</th><th>Time</th></tr></thead><tbody></tbody></table>"), display_id=True)
table_rows = ""

for epoch in range(start_epoch, NUM_EPOCHS):
    epoch_start = time.time()
    if time.time() > deadline:
        print(f"\n\u23f0 11.5 hour time budget reached! Saving and stopping gracefully.")
        break

    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, scaler, DEVICE)
    val_loss, miou = validate_one_epoch(model, val_loader, criterion, DEVICE)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["miou"].append(miou)

    epoch_duration = time.time() - epoch_start
    epoch_times.append(epoch_duration)
    avg_epoch_time = sum(epoch_times) / len(epoch_times)
    eta_seconds = int(avg_epoch_time * (NUM_EPOCHS - (epoch + 1)))
    eta_str = str(datetime.timedelta(seconds=eta_seconds))

    progress_html.update(HTML(f"<div><progress value='{epoch+1}' max='{NUM_EPOCHS}' style='width:300px; height:20px; vertical-align: middle;'></progress> [{epoch+1}/{NUM_EPOCHS} &lt; ETA: {eta_str}]</div>"))
    table_rows += f"<tr><td>{epoch+1}</td><td>{train_loss:.4f}</td><td>{val_loss:.4f}</td><td>{miou:.4f}</td><td>{epoch_duration:.1f}s</td></tr>"
    table_html.update(HTML(f"<table border='1' class='dataframe'><thead><tr style='text-align: left;'><th>Epoch</th><th>Train Loss</th><th>Val Loss</th><th>mIoU</th><th>Time</th></tr></thead><tbody>{table_rows}</tbody></table>"))

    if miou > best_miou:
        best_miou = miou
        torch.save(model.state_dict(), "/kaggle/working/best_model.pth")

    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "best_miou": best_miou,
    }, f"/kaggle/working/checkpoints/unet_epoch_{epoch}.pth")

    # ── save history every epoch (not just at the end) so a mid-run crash doesn't lose it ──
    with open("/kaggle/working/training_history.json", "w") as f:
        json.dump(history, f, indent=2)

print(f"\n\u2713 Training complete. Best mIoU: {best_miou:.4f}")


Downloading: "https://download.pytorch.org/models/resnet34-333f7ec4.pth" to /root/.cache/torch/hub/checkpoints/resnet34-333f7ec4.pth
100%|██████████| 83.3M/83.3M [00:00<00:00, 316MB/s]
/tmp/ipykernel_58/2249474387.py:24: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()  # AMP Optimization


Epoch,Train Loss,Val Loss,mIoU,Time
1,0.8113,0.6889,0.2632,197.8s
2,0.6687,0.6417,0.2661,195.3s
3,0.6219,0.6246,0.2844,195.5s
4,0.5889,0.6235,0.2850,194.4s
5,0.5583,0.6061,0.3019,193.3s
6,0.5386,0.5574,0.3102,195.1s
7,0.5146,0.5542,0.3059,196.0s
8,0.4900,0.5497,0.3174,195.5s
9,0.4809,0.5474,0.3213,197.1s
10,0.4772,0.5422,0.3210,195.1s


Train:   0%|          | 0/363 [00:00<?, ?it/s]/tmp/ipykernel_58/2249474387.py:46: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Val:   0%|          | 0/41 [00:00<?, ?it/s]             /tmp/ipykernel_58/2249474387.py:68: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



✓ Training complete. Best mIoU: 0.3213


In [11]:
# ── Cell X: Final Evaluation & Per-Class Metrics (mIoU / F1) ───────────────

@torch.no_grad()
def compute_per_class_metrics(model, loader, num_classes, device):
    model.eval()
    intersection = torch.zeros(num_classes)
    union = torch.zeros(num_classes)
    tp = torch.zeros(num_classes)
    fp = torch.zeros(num_classes)
    fn = torch.zeros(num_classes)

    for images, masks in tqdm(loader, desc="Metrics"):
        images, masks = images.to(device), masks.to(device)
        logits = model(images)
        preds = torch.argmax(logits, dim=1)

        for c in range(num_classes):
            pred_c = (preds == c)
            true_c = (masks == c)
            intersection[c] += (pred_c & true_c).sum().item()
            union[c] += (pred_c | true_c).sum().item()
            tp[c] += (pred_c & true_c).sum().item()
            fp[c] += (pred_c & ~true_c).sum().item()
            fn[c] += (~pred_c & true_c).sum().item()

    iou = intersection / (union + 1e-8)
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)

    class_names = ["Background", "Intact", "Damaged", "Destroyed"]
    print(f"\n{'Class':<12} {'IoU':>8} {'Precision':>10} {'Recall':>8} {'F1':>8}")
    for c in range(num_classes):
        print(f"{class_names[c]:<12} {iou[c]:>8.4f} {precision[c]:>10.4f} {recall[c]:>8.4f} {f1[c]:>8.4f}")
    print(f"\nMean IoU: {iou.mean():.4f}")
    print(f"Mean F1:  {f1.mean():.4f}")

    return {"iou": iou, "precision": precision, "recall": recall, "f1": f1}

# Load best checkpoint before final eval
model.load_state_dict(torch.load("/kaggle/working/best_model.pth"))
metrics = compute_per_class_metrics(model, val_loader, NUM_CLASSES, DEVICE)


/tmp/ipykernel_58/3156770216.py:41: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("/kaggle/working/best_model.pth"))
Metrics: 100%|█████████


Class             IoU  Precision   Recall       F1
Background     0.8948     0.9948   0.8990   0.9445
Intact         0.4387     0.4839   0.8243   0.6098
Damaged        0.1783     0.1903   0.7396   0.3027
Destroyed      0.0949     0.0979   0.7595   0.1734

Mean IoU: 0.4017
Mean F1:  0.5076


In [12]:
# ── A3: Save results under distinct filenames so they don't clobber baseline/A2 ──
import shutil, json

shutil.copy("/kaggle/working/best_model.pth", "/kaggle/working/best_model_copypaste.pth")

copypaste_metrics = {
    "class_names": ["Background", "Intact", "Damaged", "Destroyed"],
    "iou": metrics["iou"].tolist(),
    "precision": metrics["precision"].tolist(),
    "recall": metrics["recall"].tolist(),
    "f1": metrics["f1"].tolist(),
    "mean_iou": metrics["iou"].mean().item(),
    "mean_f1": metrics["f1"].mean().item(),
    "instances_extracted": instance_count,
}
with open("/kaggle/working/metrics_copypaste.json", "w") as f:
    json.dump(copypaste_metrics, f, indent=2)

shutil.copy("/kaggle/working/training_history.json", "/kaggle/working/history_copypaste.json")

print("\u2713 Saved: best_model_copypaste.pth, metrics_copypaste.json, history_copypaste.json")
print(f"\u2713 Instances extracted \u2014 Damaged: {instance_count[2]}, Destroyed: {instance_count[3]}")


✓ Saved: best_model_copypaste.pth, metrics_copypaste.json, history_copypaste.json
✓ Instances extracted — Damaged: 40704, Destroyed: 16682


## Recap — what's saved after this run
- `best_model_copypaste.pth` — model weights from the copy-paste run
- `metrics_copypaste.json` — per-class IoU/Precision/Recall/F1 + instance counts
- `history_copypaste.json` — per-epoch train/val loss + mIoU

Compare against your saved `metrics_baseline.json` and `metrics_sampler.json` for the final
A1–A3 report: look specifically at how the **Destroyed** row moves across all three runs.